# Wheat Kaggriculture Self-Play RL

Train two PPO agents against each other using the shared `KaggricultureEnv` from `env.py`. Each action is `[farmer_action_id, buy_wheat_quantity, sell_wheat_quantity]`, so the agents can move, plant, water, harvest, drop, pick up, buy, and sell during the same turn.

In [13]:
from pathlib import Path
import sys
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from env import KaggricultureEnv, decode_action, wheat_reward
from data import preprocess

In [3]:
# Validate the local Gym environment and action encoding.
base_env = KaggricultureEnv()
print('observation space:', base_env.observation_space)
print('action space:', base_env.action_space)
print('action dimensions:', base_env.action_space.nvec.tolist())
print('DROP example:', decode_action([8, 0, 0]))
print('buy/sell example:', decode_action([1, 2, 3]))
# base_env.close()

observation space: Box(-inf, inf, (1, 1, 5), float32)
action space: MultiDiscrete([10 11 11])
action dimensions: [10, 11, 11]
DROP example: {'farmer': ['DROP'], 'hands': [], 'market': []}
buy/sell example: {'farmer': ['NORTH'], 'hands': [], 'market': [['BUY_SEED', 'WHEAT', 2], ['SELL', 'WHEAT', 3]]}


## Self-play adapter

`KaggricultureEnv` normally uses a passing opponent. This adapter lets a frozen PPO model choose the opponent action from player 1's observation. The learning agent remains player 0, while both agents use the same observation encoder and action space.

In [14]:
class SelfPlayEnv(KaggricultureEnv):
    def __init__(self, opponent=None):
        super().__init__()
        self.opponent = opponent
        self.reset()

    def _opponent_action(self):
        if self.opponent is None:
            return {'farmer': ['PASS'], 'hands': [], 'market': []}

        opponent_obs = self.env.state[1].observation
        encoded = preprocess(opponent_obs, player_index=1)
        opponent_action_id, _ = self.opponent.predict(encoded, deterministic=False)
        return decode_action(opponent_action_id)

    def step(self, action_id):
        previous_obs = self._get_current_obs()
        action = decode_action(action_id)
        opponent_action = self._opponent_action()
        self.env.step([action, opponent_action])

        obs = self._get_current_obs()
        obs_array = preprocess(obs, self.player_index)
        reward = wheat_reward(previous_obs, obs, action, self.player_index)
        done = bool(getattr(self.env, 'done', False))
        self.observation_space = self.observation_space.__class__(
            low=-np.inf, high=np.inf, shape=obs_array.shape, dtype=np.float32
        )
        return obs_array, reward, done, False, {}

In [15]:
class RewardPrinterCallback(BaseCallback):
    def __init__(self, label, print_every=1000, verbose=0):
        super().__init__(verbose)
        self.label = label
        self.print_every = print_every
        self.rewards = []

    def _on_step(self):
        step_rewards = np.asarray(self.locals.get('rewards', []), dtype=np.float32)
        if step_rewards.size:
            self.rewards.extend(step_rewards.reshape(-1).tolist())

        if self.num_timesteps > 0 and self.num_timesteps % self.print_every == 0:
            recent = self.rewards[-self.print_every:]
            interval_mean = float(np.mean(recent)) if recent else 0.0
            overall_mean = float(np.mean(self.rewards)) if self.rewards else 0.0
            print(
                f'{self.label} step {self.num_timesteps}: '
                f'interval reward={interval_mean:.3f}, '
                f'overall reward={overall_mean:.3f}'
            )
        return True


# Train two agents in alternating rounds.
# Increase ROUND_STEPS for a stronger but longer training run.
ROUND_STEPS = 10_000
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

agent_a = PPO(
    'MlpPolicy',
    SelfPlayEnv(),
    verbose=1,
    n_steps=256,
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.99,
    seed=7,
)
agent_a.learn(
    total_timesteps=ROUND_STEPS,
    callback=RewardPrinterCallback('Agent A'),
)
agent_a.save(MODEL_DIR / 'wheat_agent_a')

agent_b = PPO(
    'MlpPolicy',
    SelfPlayEnv(opponent=agent_a),
    verbose=1,
    n_steps=256,
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.99,
    seed=8,
)
agent_b.learn(
    total_timesteps=ROUND_STEPS,
    callback=RewardPrinterCallback('Agent B'),
)
agent_b.save(MODEL_DIR / 'wheat_agent_b')

# Give agent A a second round against the improved agent B.
agent_a.set_env(SelfPlayEnv(opponent=agent_b))
agent_a.learn(
    total_timesteps=ROUND_STEPS,
    callback=RewardPrinterCallback('Agent A final'),
)
agent_a.save(MODEL_DIR / 'wheat_agent_a_final')

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------
| time/              |     |
|    fps             | 381 |
|    iterations      | 1   |
|    time_elapsed    | 0   |
|    total_timesteps | 256 |
----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 308         |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 512         |
| train/                  |             |
|    approx_kl            | 0.019833863 |
|    clip_fraction        | 0.198       |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.09       |
|    explained_variance   | -0.198      |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0983      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0336     |
|    value_loss           | 0.4

## Evaluate the learned agents

The evaluation uses the final agent A against agent B. The reported reward is the event-based wheat reward from `env.py`, not the final bank balance.

In [8]:
evaluation_env = SelfPlayEnv(opponent=agent_b)
mean_reward, std_reward = evaluate_policy(
    agent_a,
    evaluation_env,
    n_eval_episodes=3,
    deterministic=True,
)
print(f'mean episode reward: {mean_reward:.2f} +/- {std_reward:.2f}')
# evaluation_env.close()

mean episode reward: 1.00 +/- 0.00


In [16]:
# Optional: reload the saved models in a fresh session.
loaded_a = PPO.load(MODEL_DIR / 'wheat_agent_a_final')
loaded_b = PPO.load(MODEL_DIR / 'wheat_agent_b')
print('saved models loaded:', loaded_a.policy.__class__.__name__, loaded_b.policy.__class__.__name__)

saved models loaded: ActorCriticPolicy ActorCriticPolicy
